In [143]:
'''
conda activate aicam

pip install playwright pandas pyarrow
python -m playwright install chromium
'''

'\nconda activate aicam\n\npip install playwright pandas pyarrow\npython -m playwright install chromium\n'

In [4]:
from pathlib import Path
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
from urllib.parse import urlencode

import asyncio
import hashlib

import pandas as pd

from playwright.async_api import async_playwright

In [5]:
# NAVER 접근 제한 전용 제외
class NaverAccessBlockedError(RuntimeError):
    pass

In [6]:
KST = ZoneInfo("Asia/Seoul")

BASE_DIR = Path("news_data")
BASE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [7]:
SEARCH_QUERIES = {
    "한미반도체": [
        "한미반도체",

        # 핵심 사업
        "한미반도체 HBM",
        "한미반도체 TC 본더",
        "한미반도체 반도체 장비",

        # 기업 이벤트
        "한미반도체 수주",
        "한미반도체 공급",
        "한미반도체 계약",
        "한미반도체 실적",
        "한미반도체 투자",
        "한미반도체 증설",

        # 주요 관계
        "한미반도체 SK하이닉스",
    ],

    "SK하이닉스": [
        "SK하이닉스",

        # 핵심 사업
        "SK하이닉스 HBM",
        "SK하이닉스 DRAM",
        "SK하이닉스 NAND",
        "SK하이닉스 메모리",
        "SK하이닉스 AI 반도체",

        # 기업 이벤트
        "SK하이닉스 수주",
        "SK하이닉스 공급",
        "SK하이닉스 계약",
        "SK하이닉스 실적",
        "SK하이닉스 투자",
        "SK하이닉스 증설",

        # 주요 관계 / 규제
        "SK하이닉스 엔비디아",
        "SK하이닉스 수출 규제",
    ],
}

In [8]:
CASES = {
    "hanmi_20260612": {
        "company": "한미반도체",

        "collection_start": date(2026, 5, 12),
        "collection_end": date(2026, 6, 12),

        "event_start": datetime(
            2026, 6, 11, 15, 30, 0,
            tzinfo=KST,
        ),

        "event_end": datetime(
            2026, 6, 12, 15, 29, 59,
            tzinfo=KST,
        ),
    },

    "skhynix_20260304": {
        "company": "SK하이닉스",

        "collection_start": date(2026, 2, 1),
        "collection_end": date(2026, 3, 4),

        "event_start": datetime(
            2026, 3, 3, 15, 30, 0,
            tzinfo=KST,
        ),

        "event_end": datetime(
            2026, 3, 4, 15, 29, 59,
            tzinfo=KST,
        ),
    },
}

In [9]:
def iter_dates(
    start_date: date,
    end_date: date,
):
    current = start_date

    while current <= end_date:
        yield current
        current += timedelta(days=1)

In [10]:
def build_naver_news_url(
    query: str,
    start_date: date,
    end_date: date,
):
    start_dot = start_date.strftime("%Y.%m.%d")
    end_dot = end_date.strftime("%Y.%m.%d")

    start_compact = start_date.strftime("%Y%m%d")
    end_compact = end_date.strftime("%Y%m%d")

    params = {
        "where": "news",
        "ssc": "tab.news.all",
        "query": query,

        # 최신순
        "sort": "1",

        # 기간 직접 설정
        "pd": "3",
        "ds": start_dot,
        "de": end_dot,

        "nso": (
            f"so:dd,"
            f"p:from{start_compact}to{end_compact}"
        ),

        # 전체 언론사
        "mynews": "0",
        "office_category": "0",
        "office_section_code": "0",
        "office_type": "0",

        "photo": "0",
        "service_area": "0",
    }

    return (
        "https://search.naver.com/search.naver?"
        + urlencode(params)
    )

In [11]:
print(
    build_naver_news_url(
        query="한미반도체",
        start_date=date(2026, 6, 12),
        end_date=date(2026, 6, 12),
    )
)

https://search.naver.com/search.naver?where=news&ssc=tab.news.all&query=%ED%95%9C%EB%AF%B8%EB%B0%98%EB%8F%84%EC%B2%B4&sort=1&pd=3&ds=2026.06.12&de=2026.06.12&nso=so%3Add%2Cp%3Afrom20260612to20260612&mynews=0&office_category=0&office_section_code=0&office_type=0&photo=0&service_area=0


In [12]:
NAVER_NEWS_CONTAINER_SELECTOR = (
    "div.fds-news-item-list-tab"
)

NAVER_NEWS_ITEM_SELECTOR = (
    ":scope > "
    "div.sds-comps-vertical-layout.sds-comps-full-layout"
)

NAVER_HEADLINE_SELECTOR = (
    "span.sds-comps-text-type-headline1"
)

NAVER_PUBLISHER_SELECTOR = (
    ".sds-comps-profile-info-title-text"
)

In [13]:
async def check_naver_access_block(page):
    body_text = await (
        page.locator("body")
        .inner_text()
    )

    blocked_words = [
        "비정상적인 접근",
        "자동입력 방지",
        "접근이 제한",
    ]

    return any(
        word in body_text
        for word in blocked_words
    )

In [14]:
async def scroll_to_naver_news_end(
    page,
    step_ratio=0.8,
    scroll_wait_ms=700,
    bottom_wait_ms=1500,
    stable_bottom_rounds=5,
    max_scrolls=1000,
):
    """
    NAVER 뉴스 검색 결과를 점진적으로 끝까지 로딩합니다.

    종료 판단:
    - 뉴스 카드 개수 사용 X
    - URL 개수 사용 X
    - 페이지 하단에서 scrollHeight가
      stable_bottom_rounds회 연속 증가하지 않으면 종료
    """

    stable_rounds = 0

    for scroll_no in range(
        1,
        max_scrolls + 1,
    ):
        state = await page.evaluate(
            """
            () => ({
                scrollY: window.scrollY,
                viewportHeight: window.innerHeight,
                scrollHeight:
                    document.scrollingElement.scrollHeight
            })
            """
        )

        scroll_y = state["scrollY"]
        viewport_height = state["viewportHeight"]
        scroll_height = state["scrollHeight"]

        reached_bottom = (
            scroll_y + viewport_height
            >= scroll_height - 5
        )

        print(
            f"[scroll {scroll_no:03d}] "
            f"y={scroll_y:.0f} | "
            f"height={scroll_height:.0f}"
        )

        # --------------------------------
        # 아직 페이지 하단이 아닌 경우
        # --------------------------------
        if not reached_bottom:
            stable_rounds = 0

            step = int(
                viewport_height
                * step_ratio
            )

            await page.mouse.wheel(
                0,
                step,
            )

            await page.wait_for_timeout(
                scroll_wait_ms
            )

            continue

        # --------------------------------
        # 하단 도달
        # --------------------------------
        print(
            "          → 페이지 하단 도달"
        )

        await page.wait_for_timeout(
            bottom_wait_ms
        )

        new_height = await page.evaluate(
            """
            () =>
                document.scrollingElement.scrollHeight
            """
        )

        # 새로운 결과가 추가됨
        if new_height > scroll_height:
            print(
                f"          → 추가 로딩 "
                f"{scroll_height} → {new_height}"
            )

            stable_rounds = 0
            continue

        # 추가 결과 없음
        stable_rounds += 1

        print(
            f"          → 추가 로딩 없음 "
            f"({stable_rounds}/"
            f"{stable_bottom_rounds})"
        )

        if (
            stable_rounds
            >= stable_bottom_rounds
        ):
            print(
                "[DONE] 페이지 하단에서 "
                f"{stable_bottom_rounds}회 연속 "
                "추가 로딩 없음"
            )

            return

        # 하단에서 wheel 이벤트를 추가로 발생시켜
        # 다음 batch가 존재하는지 다시 확인
        await page.mouse.wheel(
            0,
            int(viewport_height * 0.3),
        )

    raise RuntimeError(
        f"max_scrolls={max_scrolls}에 도달했습니다."
    )

In [15]:
async def extract_raw_naver_news_items(
    page,
):
    """
    최종 DOM에 존재하는 뉴스 item을 그대로 추출합니다.

    여기서는 중복 제거를 하지 않습니다.
    """

    return await page.evaluate(
        """
        ([
            containerSelector,
            itemSelector,
            headlineSelector,
            publisherSelector
        ]) => {

            const containers = Array.from(
                document.querySelectorAll(
                    containerSelector
                )
            );

            const results = [];

            let domRank = 1;


            containers.forEach(
                (container, containerIndex) => {

                    const items = Array.from(
                        container.querySelectorAll(
                            itemSelector
                        )
                    );


                    items.forEach(
                        (item, itemIndex) => {

                            // =====================
                            // 제목 + URL
                            // =====================

                            const headline =
                                item.querySelector(
                                    headlineSelector
                                );

                            if (!headline) {
                                return;
                            }


                            const anchor =
                                headline.closest("a");

                            if (
                                !anchor
                                || !anchor.href
                            ) {
                                return;
                            }


                            const title =
                                headline.innerText
                                    .replace(
                                        /새 창 열림/g,
                                        ""
                                    )
                                    .trim();


                            // =====================
                            // 언론사
                            // =====================

                            const publisherEl =
                                item.querySelector(
                                    publisherSelector
                                );


                            const publisherName =
                                publisherEl
                                ? publisherEl
                                    .innerText
                                    .replace(
                                        /새 창 열림/g,
                                        ""
                                    )
                                    .trim()
                                : null;


                            // =====================
                            // NAVER 검색 snippet
                            // =====================

                            const bodyCandidates =
                                Array.from(
                                    item.querySelectorAll(
                                        "span.sds-comps-text-type-body1"
                                    )
                                )
                                .map(
                                    el =>
                                        el.innerText
                                            .replace(
                                                /새 창 열림/g,
                                                ""
                                            )
                                            .trim()
                                )
                                .filter(
                                    text =>
                                        text
                                        && text !== title
                                        && text !== publisherName
                                        && text !== "네이버뉴스"
                                );


                            bodyCandidates.sort(
                                (a, b) =>
                                    b.length - a.length
                            );


                            const description =
                                bodyCandidates.length > 0
                                ? bodyCandidates[0]
                                : null;


                            // =====================
                            // 화면상의 문서 위치
                            // =====================

                            const rect =
                                item.getBoundingClientRect();


                            results.push({
                                dom_rank:
                                    domRank++,

                                container_index:
                                    containerIndex,

                                item_index_in_container:
                                    itemIndex + 1,

                                absolute_y:
                                    rect.top
                                    + window.scrollY,

                                publisher_name:
                                    publisherName,

                                title:
                                    title,

                                description:
                                    description,

                                original_link:
                                    anchor.href
                            });
                        }
                    );
                }
            );


            return results;
        }
        """,
        [
            NAVER_NEWS_CONTAINER_SELECTOR,
            NAVER_NEWS_ITEM_SELECTOR,
            NAVER_HEADLINE_SELECTOR,
            NAVER_PUBLISHER_SELECTOR,
        ]
    )

In [16]:
def clean_search_hits(
    raw_df: pd.DataFrame,
):
    """
    같은 query + 같은 날짜의 NAVER 검색 페이지 안에서

    publisher_name
    + title
    + original_link

    가 모두 동일한 경우 하나만 남깁니다.
    """

    if raw_df.empty:
        return raw_df.copy()


    cleaned_df = (
        raw_df
        .sort_values("dom_rank")
        .drop_duplicates(
            subset=[
                "publisher_name",
                "title",
                "original_link",
            ],
            keep="first",
        )
        .reset_index(drop=True)
        .copy()
    )


    # 분석용 검색 순위 재부여
    cleaned_df["search_rank"] = range(
        1,
        len(cleaned_df) + 1,
    )


    return cleaned_df

In [17]:
async def test_naver_search(
    headless=False,
):
    query = "한미반도체"
    target_date = date(
        2026,
        6,
        12,
    )


    url = build_naver_news_url(
        query=query,
        start_date=target_date,
        end_date=target_date,
    )


    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=headless
        )

        context = await browser.new_context(
            locale="ko-KR",
            timezone_id="Asia/Seoul",
        )

        page = await context.new_page()


        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=30_000,
        )


        if await check_naver_access_block(
            page
        ):
            raise RuntimeError(
                "NAVER 접근 제한이 감지되었습니다."
            )


        # 1. 점진적 스크롤
        await scroll_to_naver_news_end(
            page
        )


        # 2. 최종 DOM 추출
        raw_rows = (
            await extract_raw_naver_news_items(
                page
            )
        )


        await context.close()
        await browser.close()


    raw_df = pd.DataFrame(
        raw_rows
    )


    # 3. 내용 기준 중복 제거
    search_df = clean_search_hits(
        raw_df
    )


    return raw_df, search_df

In [18]:
raw_df, search_df = (
    await test_naver_search()
)

[scroll 001] y=0 | height=2594
[scroll 002] y=576 | height=2594
[scroll 003] y=1152 | height=4324
[scroll 004] y=1728 | height=4324
[scroll 005] y=2304 | height=4324
[scroll 006] y=2880 | height=6052
[scroll 007] y=3456 | height=6052
[scroll 008] y=4032 | height=6052
[scroll 009] y=4608 | height=7782
[scroll 010] y=5184 | height=7782
[scroll 011] y=5760 | height=7782
[scroll 012] y=6336 | height=9512
[scroll 013] y=6912 | height=9512
[scroll 014] y=7488 | height=9512
[scroll 015] y=8064 | height=11240
[scroll 016] y=8640 | height=11240
[scroll 017] y=9216 | height=11240
[scroll 018] y=9792 | height=12970
[scroll 019] y=10368 | height=12970
[scroll 020] y=10944 | height=12970
[scroll 021] y=11520 | height=14700
[scroll 022] y=12096 | height=14700
[scroll 023] y=12672 | height=14700
[scroll 024] y=13248 | height=16430
[scroll 025] y=13824 | height=16430
[scroll 026] y=14400 | height=16430
[scroll 027] y=14976 | height=18160
[scroll 028] y=15552 | height=18160
[scroll 029] y=16128 | heigh

In [19]:
print(
    "Raw DOM item 수:",
    len(raw_df)
)

print(
    "Search Hit 수:",
    len(search_df)
)

print(
    "제거된 완전 중복 수:",
    len(raw_df) - len(search_df)
)

print(
    "Search Hit의 고유 original_link:",
    search_df["original_link"].nunique()
)

Raw DOM item 수: 132
Search Hit 수: 130
제거된 완전 중복 수: 2
Search Hit의 고유 original_link: 130


In [20]:
duplicate_raw_df = (
    raw_df[
        raw_df.duplicated(
            subset=[
                "publisher_name",
                "title",
                "original_link",
            ],
            keep=False,
        )
    ]
    .sort_values(
        [
            "original_link",
            "dom_rank",
        ]
    )
)

duplicate_raw_df[
    [
        "dom_rank",
        "container_index",
        "item_index_in_container",
        "publisher_name",
        "title",
        "original_link",
    ]
]

,dom_rank,container_index,item_index_in_container,publisher_name,title,original_link
79,80,7,10,뉴스핌,"[특징주] 한미반도체, 스페이스X 500억 투자 소식에 14% 급등",https://www.newspim.com/news/view/20260612000376
80,81,8,1,뉴스핌,"[특징주] 한미반도체, 스페이스X 500억 투자 소식에 14% 급등",https://www.newspim.com/news/view/20260612000376
109,110,10,10,더스탁,"한미반도체, 스페이스X에 500억 투자…성장축 '우주'로 확장",https://www.the-stock.kr/news/articleView.html...
110,111,11,1,더스탁,"한미반도체, 스페이스X에 500억 투자…성장축 '우주'로 확장",https://www.the-stock.kr/news/articleView.html...


In [21]:
def make_search_hit_id(
    case_id: str,
    query: str,
    search_date: str,
    search_rank: int,
    original_link: str,
):
    value = (
        f"{case_id}|"
        f"{query}|"
        f"{search_date}|"
        f"{search_rank}|"
        f"{original_link}"
    )

    digest = hashlib.sha1(
        value.encode("utf-8")
    ).hexdigest()

    return (
        f"H_{digest[:16]}"
    )

In [22]:
def load_collection_log(
    path: Path,
):
    columns = [
        "query",
        "search_date",
        "status",

        "raw_item_count",
        "search_hit_count",
        "removed_duplicate_count",

        "error_type",
        "error_message",

        "collected_at",
    ]

    if path.exists():
        df = pd.read_parquet(path)

        # 기존 parquet에 새 컬럼이 없으면 추가
        for column in columns:
            if column not in df.columns:
                df[column] = None

        return df[columns]

    return pd.DataFrame(
        columns=columns
    )

In [23]:
async def collect_search_hits(
    case_id: str,
    case: dict,
    headless=False,
):
    company = case["company"]

    queries = SEARCH_QUERIES[
        company
    ]


    # =====================================
    # 저장 경로
    # =====================================

    case_dir = (
        BASE_DIR / case_id
    )

    case_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    raw_path = (
        case_dir
        / "raw_search_items.parquet"
    )

    hits_path = (
        case_dir
        / "search_hits.parquet"
    )

    log_path = (
        case_dir
        / "collection_log.parquet"
    )


    # =====================================
    # 기존 파일
    # =====================================

    if raw_path.exists():
        raw_all_df = pd.read_parquet(
            raw_path
        )
    else:
        raw_all_df = pd.DataFrame()


    if hits_path.exists():
        hits_all_df = pd.read_parquet(
            hits_path
        )
    else:
        hits_all_df = pd.DataFrame()


    log_df = load_collection_log(
        log_path
    )

    # =====================================
    # Browser
    # =====================================

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=headless
        )

        context = await browser.new_context(
            locale="ko-KR",
            timezone_id="Asia/Seoul",
        )

        page = await context.new_page()


        # =================================
        # Query × Date
        # =================================

        for query in queries:

            for target_date in iter_dates(
                case["collection_start"],
                case["collection_end"],
            ):
                date_str = (
                    target_date.isoformat()
                )


                # -------------------------
                # 이미 성공한 작업 skip
                # -------------------------

                already_done = (
                    (
                        log_df["query"]
                        == query
                    )
                    &
                    (
                        log_df["search_date"]
                        == date_str
                    )
                    &
                    (
                        log_df["status"]
                        == "SUCCESS"
                    )
                ).any()


                if already_done:
                    print(
                        "[SKIP]",
                        query,
                        date_str,
                    )

                    continue


                print()
                print("=" * 100)

                print(
                    company,
                    "|",
                    query,
                    "|",
                    date_str,
                )

                print("=" * 100)

                # -------------------------------------------------
                # URL 생성
                # -------------------------------------------------

                url = build_naver_news_url(
                    query=query,
                    start_date=target_date,
                    end_date=target_date,
                )


                # -------------------------------------------------
                # 현재 작업 초기 상태
                # -------------------------------------------------

                status = "FAILED"

                raw_item_count = 0
                search_hit_count = 0
                removed_duplicate_count = 0

                error_type = None
                error_message = None


                # =================================================
                # 실제 수집
                # =================================================
                try:

                    # =====================
                    # NAVER 접속
                    # =====================

                    await page.goto(
                        url,
                        wait_until="domcontentloaded",
                        timeout=30_000,
                    )

                    # ---------------------------------------------
                    # 접근 제한 확인
                    # ---------------------------------------------
                    if await check_naver_access_block(
                        page
                    ):
                        raise NaverAccessBlockedError(
                            "NAVER 접근 제한 감지"
                        )


                    # =====================
                    # 점진적 스크롤
                    # =====================

                    await scroll_to_naver_news_end(
                        page
                    )


                    # =====================
                    # 최종 DOM에서 Raw item 추출
                    # =====================

                    raw_rows = (
                        await extract_raw_naver_news_items(
                            page
                        )
                    )


                    page_raw_df = pd.DataFrame(
                        raw_rows
                    )


                    raw_item_count = len(
                        page_raw_df
                    )


                    # =====================
                    # 검색 결과 정제
                    # 동일 카드 제거
                    #
                    # publisher_name + title + original_link
                    #
                    # =====================

                    page_hits_df = (
                        clean_search_hits(
                            page_raw_df
                        )
                    )


                    search_hit_count = len(
                        page_hits_df
                    )

                    removed_duplicate_count = (
                        raw_item_count
                        - search_hit_count
                    )


                    print(
                        "Raw DOM item:",
                        raw_item_count
                    )

                    print(
                        "Search Hit:",
                        search_hit_count
                    )

                    print(
                        "완전 중복 제거:",
                        removed_duplicate_count
                    )


                    collected_at = (
                        datetime.now(
                            KST
                        ).isoformat()
                    )


                    # =====================
                    # metadata 추가
                    # =====================

                    if not page_raw_df.empty:

                        page_raw_df.insert(
                            0,
                            "case_id",
                            case_id,
                        )

                        page_raw_df.insert(
                            1,
                            "company",
                            company,
                        )

                        page_raw_df.insert(
                            2,
                            "query",
                            query,
                        )

                        page_raw_df.insert(
                            3,
                            "search_date",
                            date_str,
                        )

                        page_raw_df[
                            "collected_at"
                        ] = collected_at


                    if not page_hits_df.empty:

                        page_hits_df.insert(
                            0,
                            "case_id",
                            case_id,
                        )

                        page_hits_df.insert(
                            1,
                            "company",
                            company,
                        )

                        page_hits_df.insert(
                            2,
                            "query",
                            query,
                        )

                        page_hits_df.insert(
                            3,
                            "search_date",
                            date_str,
                        )

                        # ---------------------------------------------
                        # Search Hit ID
                        # ---------------------------------------------
                        page_hits_df[
                            "search_hit_id"
                        ] = page_hits_df.apply(
                            lambda row:
                                make_search_hit_id(
                                    case_id=case_id,
                                    query=query,
                                    search_date=date_str,
                                    search_rank=row[
                                        "search_rank"
                                    ],
                                    original_link=row[
                                        "original_link"
                                    ],
                                ),
                            axis=1,
                        )


                        page_hits_df[
                            "collected_at"
                        ] = collected_at


                    # =================================================
                    # 재수집 시 기존 행 제거
                    #
                    # 기존 동일 query/date 데이터 제거
                    #
                    # 실패 후 재수집한 경우
                    # 기존 데이터를 새 결과로 교체하기 위함
                    # =================================================

                    if not raw_all_df.empty:

                        raw_all_df = (
                            raw_all_df[
                                ~(
                                    (
                                        raw_all_df[
                                            "query"
                                        ]
                                        == query
                                    )
                                    &
                                    (
                                        raw_all_df[
                                            "search_date"
                                        ]
                                        == date_str
                                    )
                                )
                            ]
                            .reset_index(
                                drop=True
                            )
                        )


                    if not hits_all_df.empty:

                        hits_all_df = (
                            hits_all_df[
                                ~(
                                    (
                                        hits_all_df[
                                            "query"
                                        ]
                                        == query
                                    )
                                    &
                                    (
                                        hits_all_df[
                                            "search_date"
                                        ]
                                        == date_str
                                    )
                                )
                            ]
                            .reset_index(
                                drop=True
                            )
                        )


                    # =====================
                    # 새 결과 append
                    # =====================

                    if not page_raw_df.empty:

                        raw_all_df = pd.concat(
                            [
                                raw_all_df,
                                page_raw_df,
                            ],
                            ignore_index=True,
                        )


                    if not page_hits_df.empty:

                        hits_all_df = pd.concat(
                            [
                                hits_all_df,
                                page_hits_df,
                            ],
                            ignore_index=True,
                        )


                    status = "SUCCESS"

                # =====================================================
                # NAVER 접근 제한
                # =====================================================

                except NaverAccessBlockedError as e:

                    error_type = type(e).__name__
                    error_message = str(e)
                    

                    print(
                        "[ACCESS BLOCKED]",
                        error_message,
                    )

                    print(
                        "현재 날짜는 FAILED로 기록하고 "
                        "다음 날짜로 계속 진행합니다."
                    )

                # =====================================================
                # 일반 오류
                # =====================================================

                except Exception as e:

                    error_type = type(e).__name__
                    error_message = str(e)

                    print(
                        "[ERROR]",
                        error_type,
                        error_message,
                    )


                # =========================
                # 현재까지의 데이터 Checkpoint
                # =========================

                if not raw_all_df.empty:

                    raw_all_df.to_parquet(
                        raw_path,
                        index=False,
                    )


                if not hits_all_df.empty:

                    hits_all_df.to_parquet(
                        hits_path,
                        index=False,
                    )


                # =========================
                # Log 교체
                # =========================
                # 이전 동일 query/date 로그 제거

                log_df = (
                    log_df[
                        ~(
                            (
                                log_df["query"]
                                == query
                            )
                            &
                            (
                                log_df["search_date"]
                                == date_str
                            )
                        )
                    ]
                    .reset_index(
                        drop=True
                    )
                )


                log_row = pd.DataFrame([
                    {
                        "query":
                            query,

                        "search_date":
                            date_str,

                        "status":
                            status,

                        "raw_item_count":
                            raw_item_count,

                        "search_hit_count":
                            search_hit_count,

                        "removed_duplicate_count":
                            removed_duplicate_count,

                        "error_type":
                            error_type,

                        "error_message":
                            error_message,

                        "collected_at":
                            datetime.now(
                                KST
                            ).isoformat(),
                    }
                ])


                log_df = pd.concat(
                    [
                        log_df,
                        log_row,
                    ],
                    ignore_index=True,
                )


                log_df.to_parquet(
                    log_path,
                    index=False,
                )

                # =====================================================
                # 다음 검색 요청까지 간격
                # =====================================================

                await asyncio.sleep(
                    3.0
                )


        # =============================================================
        # Browser 종료
        # =============================================================

        await context.close()
        await browser.close()


    return raw_all_df, hits_all_df, log_df

In [24]:
hanmi_raw, hanmi_hits, hanmi_log = (
    await collect_search_hits(
        case_id="hanmi_20260612",
        case=CASES[
            "hanmi_20260612"
        ],
        headless=False,
    )
)

[SKIP] 한미반도체 2026-05-12
[SKIP] 한미반도체 2026-05-13
[SKIP] 한미반도체 2026-05-14
[SKIP] 한미반도체 2026-05-15
[SKIP] 한미반도체 2026-05-16
[SKIP] 한미반도체 2026-05-17
[SKIP] 한미반도체 2026-05-18
[SKIP] 한미반도체 2026-05-19
[SKIP] 한미반도체 2026-05-20
[SKIP] 한미반도체 2026-05-21
[SKIP] 한미반도체 2026-05-22
[SKIP] 한미반도체 2026-05-23
[SKIP] 한미반도체 2026-05-24
[SKIP] 한미반도체 2026-05-25
[SKIP] 한미반도체 2026-05-26
[SKIP] 한미반도체 2026-05-27
[SKIP] 한미반도체 2026-05-28
[SKIP] 한미반도체 2026-05-29
[SKIP] 한미반도체 2026-05-30
[SKIP] 한미반도체 2026-05-31
[SKIP] 한미반도체 2026-06-01
[SKIP] 한미반도체 2026-06-02
[SKIP] 한미반도체 2026-06-03
[SKIP] 한미반도체 2026-06-04
[SKIP] 한미반도체 2026-06-05
[SKIP] 한미반도체 2026-06-06
[SKIP] 한미반도체 2026-06-07
[SKIP] 한미반도체 2026-06-08
[SKIP] 한미반도체 2026-06-09
[SKIP] 한미반도체 2026-06-10
[SKIP] 한미반도체 2026-06-11
[SKIP] 한미반도체 2026-06-12
[SKIP] 한미반도체 HBM 2026-05-12
[SKIP] 한미반도체 HBM 2026-05-13
[SKIP] 한미반도체 HBM 2026-05-14
[SKIP] 한미반도체 HBM 2026-05-15
[SKIP] 한미반도체 HBM 2026-05-16
[SKIP] 한미반도체 HBM 2026-05-17
[SKIP] 한미반도체 HBM 2026-05-18
[SKIP] 한미반도체 HBM 2026-05-19
[SKIP] 한

In [25]:
print(
    "전체 Raw DOM Items:",
    len(hanmi_raw)
)

print(
    "전체 Search Hits:",
    len(hanmi_hits)
)

print(
    "전체 제거 중복:",
    len(hanmi_raw)
    - len(hanmi_hits)
)

전체 Raw DOM Items: 5099
전체 Search Hits: 5086
전체 제거 중복: 13


In [26]:
hanmi_log[
    [
        "query",
        "search_date",
        "status",
        "raw_item_count",
        "search_hit_count",
        "removed_duplicate_count",
    ]
]

,query,search_date,status,raw_item_count,search_hit_count,removed_duplicate_count
0,한미반도체,2026-05-12,SUCCESS,34,34,0
1,한미반도체,2026-05-13,SUCCESS,54,54,0
2,한미반도체,2026-05-14,SUCCESS,20,20,0
3,한미반도체,2026-05-15,SUCCESS,80,79,1
4,한미반도체,2026-05-16,SUCCESS,9,9,0
...,...,...,...,...,...,...
347,한미반도체 SK하이닉스,2026-06-08,SUCCESS,76,76,0
348,한미반도체 SK하이닉스,2026-06-09,SUCCESS,34,34,0
349,한미반도체 SK하이닉스,2026-06-10,SUCCESS,12,12,0
350,한미반도체 SK하이닉스,2026-06-11,SUCCESS,30,30,0


In [27]:
hanmi_log[
    hanmi_log["status"]
    != "SUCCESS"
]

,query,search_date,status,raw_item_count,search_hit_count,removed_duplicate_count,error_type,error_message,collected_at


In [28]:
skhynix_raw, skhynix_hits, skhynix_log = (
    await collect_search_hits(
        case_id="skhynix_20260304",
        case=CASES[
            "skhynix_20260304"
        ],
        headless=False,
    )
)

[SKIP] SK하이닉스 2026-02-01
[SKIP] SK하이닉스 2026-02-02
[SKIP] SK하이닉스 2026-02-03
[SKIP] SK하이닉스 2026-02-04
[SKIP] SK하이닉스 2026-02-05
[SKIP] SK하이닉스 2026-02-06
[SKIP] SK하이닉스 2026-02-07
[SKIP] SK하이닉스 2026-02-08
[SKIP] SK하이닉스 2026-02-09
[SKIP] SK하이닉스 2026-02-10
[SKIP] SK하이닉스 2026-02-11
[SKIP] SK하이닉스 2026-02-12
[SKIP] SK하이닉스 2026-02-13
[SKIP] SK하이닉스 2026-02-14
[SKIP] SK하이닉스 2026-02-15
[SKIP] SK하이닉스 2026-02-16
[SKIP] SK하이닉스 2026-02-17
[SKIP] SK하이닉스 2026-02-18
[SKIP] SK하이닉스 2026-02-19
[SKIP] SK하이닉스 2026-02-20
[SKIP] SK하이닉스 2026-02-21
[SKIP] SK하이닉스 2026-02-22
[SKIP] SK하이닉스 2026-02-23
[SKIP] SK하이닉스 2026-02-24
[SKIP] SK하이닉스 2026-02-25
[SKIP] SK하이닉스 2026-02-26
[SKIP] SK하이닉스 2026-02-27
[SKIP] SK하이닉스 2026-02-28
[SKIP] SK하이닉스 2026-03-01
[SKIP] SK하이닉스 2026-03-02
[SKIP] SK하이닉스 2026-03-03
[SKIP] SK하이닉스 2026-03-04
[SKIP] SK하이닉스 HBM 2026-02-01
[SKIP] SK하이닉스 HBM 2026-02-02
[SKIP] SK하이닉스 HBM 2026-02-03
[SKIP] SK하이닉스 HBM 2026-02-04
[SKIP] SK하이닉스 HBM 2026-02-05
[SKIP] SK하이닉스 HBM 2026-02-06
[SKIP] SK하이닉스 HBM 2026-02-

In [29]:
print(
    "전체 Raw DOM Items:",
    len(skhynix_raw)
)

print(
    "전체 Search Hits:",
    len(skhynix_hits)
)

print(
    "전체 제거 중복:",
    len(skhynix_raw)
    - len(skhynix_hits)
)

전체 Raw DOM Items: 65996
전체 Search Hits: 64821
전체 제거 중복: 1175


In [30]:
skhynix_log[
    [
        "query",
        "search_date",
        "status",
        "raw_item_count",
        "search_hit_count",
        "removed_duplicate_count",
    ]
]

,query,search_date,status,raw_item_count,search_hit_count,removed_duplicate_count
0,SK하이닉스,2026-02-01,SUCCESS,208,207,1
1,SK하이닉스,2026-02-02,SUCCESS,838,801,37
2,SK하이닉스,2026-02-03,SUCCESS,917,877,40
3,SK하이닉스,2026-02-04,SUCCESS,859,827,32
4,SK하이닉스,2026-02-05,SUCCESS,824,804,20
...,...,...,...,...,...,...
443,SK하이닉스 수출 규제,2026-03-03,SUCCESS,3,3,0
444,SK하이닉스 수출 규제,2026-03-04,SUCCESS,11,11,0
445,SK하이닉스 수출 규제,2026-02-03,FAILED,0,0,0
446,SK하이닉스 수출 규제,2026-02-04,FAILED,0,0,0


In [31]:
skhynix_log[
    skhynix_log["status"]
    != "SUCCESS"
]

,query,search_date,status,raw_item_count,search_hit_count,removed_duplicate_count,error_type,error_message,collected_at
445,SK하이닉스 수출 규제,2026-02-03,FAILED,0,0,0,NaverAccessBlockedError,NAVER 접근 제한 감지,2026-10-10T02:11:35.680528+09:00
446,SK하이닉스 수출 규제,2026-02-04,FAILED,0,0,0,NaverAccessBlockedError,NAVER 접근 제한 감지,2026-10-10T02:11:39.217157+09:00
447,SK하이닉스 수출 규제,2026-02-15,FAILED,0,0,0,NaverAccessBlockedError,NAVER 접근 제한 감지,2026-10-10T02:11:42.674882+09:00


In [ ]:
# pip install requests trafilatura beautifulsoup4 python-dateutil

In [33]:
import gzip
import json
import requests
import trafilatura

from bs4 import BeautifulSoup
from dateutil import parser as dtparser
from urllib.parse import urlparse

In [34]:
NEWS_STORE_DIR = Path(
    "./stored_data/news"
)

NEWS_STORE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [35]:
def make_article_id(
    original_link: str,
):
    digest = hashlib.sha1(
        original_link.encode("utf-8")
    ).hexdigest()

    return f"A_{digest[:16]}"

In [36]:
def make_unique_articles(
    search_hits: pd.DataFrame,
):
    """
    Search Hit들을 실제 기사 단위로 변환합니다.

    original_link가 동일하면
    실제 원문 수집은 한 번만 수행합니다.
    """

    if search_hits.empty:
        return pd.DataFrame()


    # 최초 발견 metadata를 대표값으로 사용
    articles = (
        search_hits
        .sort_values(
            [
                "search_date",
                "search_rank",
            ]
        )
        .drop_duplicates(
            subset=[
                "original_link"
            ],
            keep="first",
        )
        .reset_index(drop=True)
        .copy()
    )


    articles["article_id"] = (
        articles[
            "original_link"
        ]
        .apply(
            make_article_id
        )
    )


    articles["publisher_domain"] = (
        articles[
            "original_link"
        ]
        .apply(
            lambda url:
                urlparse(url)
                .netloc
                .lower()
        )
    )


    # 같은 URL이 검색 결과에서 몇 번 발견됐는지
    hit_counts = (
        search_hits
        .groupby(
            "original_link"
        )
        .size()
        .rename(
            "search_hit_count"
        )
        .reset_index()
    )


    articles = articles.merge(
        hit_counts,
        on="original_link",
        how="left",
    )


    return articles[
        [
            "article_id",
            "case_id",
            "company",

            "publisher_name",
            "publisher_domain",

            "title",
            "description",

            "original_link",
            "search_hit_count",
        ]
    ]

In [37]:
def save_raw_html(
    html: str,
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with gzip.open(
        path,
        "wt",
        encoding="utf-8",
    ) as f:
        f.write(html)

In [38]:
def extract_article_body(
    html: str,
):
    if not html:
        return None


    body = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )


    if not body:
        return None


    body = body.strip()


    # 너무 짧으면 정상 기사 본문으로 보지 않음
    if len(body) < 100:
        return None


    return body

In [39]:
def normalize_datetime(
    value,
):
    if not value:
        return None


    try:
        dt = dtparser.parse(
            str(value)
        )


        # 한국 언론사 페이지에서 timezone이 없는 경우
        # KST로 간주
        if dt.tzinfo is None:

            dt = dt.replace(
                tzinfo=KST
            )

        else:

            dt = dt.astimezone(
                KST
            )


        return dt.isoformat()


    except Exception:
        return None

In [40]:
def find_date_published(
    obj,
):
    if isinstance(obj, dict):

        value = obj.get(
            "datePublished"
        )

        if value:
            return value


        for child in obj.values():

            result = find_date_published(
                child
            )

            if result:
                return result


    elif isinstance(obj, list):

        for child in obj:

            result = find_date_published(
                child
            )

            if result:
                return result


    return None

In [41]:
def extract_published_at(
    html: str,
):
    if not html:
        return None, None


    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # =====================================
    # META
    # =====================================

    meta_candidates = [
        (
            "property",
            "article:published_time"
        ),
        (
            "property",
            "og:published_time"
        ),
        (
            "name",
            "article:published_time"
        ),
        (
            "name",
            "pubdate"
        ),
        (
            "name",
            "publish-date"
        ),
        (
            "name",
            "date"
        ),
    ]


    for attr, value in meta_candidates:

        tag = soup.find(
            "meta",
            attrs={
                attr: value
            }
        )


        if not tag:
            continue


        normalized = normalize_datetime(
            tag.get("content")
        )


        if normalized:

            return (
                normalized,
                f"meta:{attr}={value}",
            )


    # =====================================
    # JSON-LD
    # =====================================

    for script in soup.find_all(
        "script",
        type="application/ld+json",
    ):

        raw = script.string

        if not raw:
            continue


        try:
            data = json.loads(raw)

        except Exception:
            continue


        value = find_date_published(
            data
        )


        normalized = normalize_datetime(
            value
        )


        if normalized:

            return (
                normalized,
                "jsonld:datePublished",
            )


    # =====================================
    # <time datetime="">
    # =====================================

    time_tag = soup.find(
        "time",
        attrs={
            "datetime": True
        }
    )


    if time_tag:

        normalized = normalize_datetime(
            time_tag.get(
                "datetime"
            )
        )


        if normalized:

            return (
                normalized,
                "time:datetime",
            )


    return None, None

In [42]:
REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Safari/537.36"
    )
}

In [43]:
def download_html_requests(
    session: requests.Session,
    url: str,
):
    try:

        response = session.get(
            url,
            headers=REQUEST_HEADERS,
            timeout=15,
            allow_redirects=True,
        )


        response.raise_for_status()


        # 인코딩이 명시되어 있지 않은 경우만 보완
        if not response.encoding:
            response.encoding = (
                response.apparent_encoding
            )


        return {
            "html":
                response.text,

            "final_url":
                response.url,

            "http_status":
                response.status_code,

            "fetch_method":
                "REQUESTS",

            "fetch_error":
                None,
        }


    except Exception as e:

        return {
            "html":
                None,

            "final_url":
                None,

            "http_status":
                None,

            "fetch_method":
                "REQUESTS",

            "fetch_error":
                f"{type(e).__name__}: {e}",
        }

In [44]:
async def download_html_playwright(
    page,
    url: str,
):
    try:

        response = await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=30_000,
        )


        await page.wait_for_timeout(
            1500
        )


        html = await page.content()


        return {
            "html":
                html,

            "final_url":
                page.url,

            "http_status":
                (
                    response.status
                    if response
                    else None
                ),

            "fetch_method":
                "PLAYWRIGHT",

            "fetch_error":
                None,
        }


    except Exception as e:

        return {
            "html":
                None,

            "final_url":
                None,

            "http_status":
                None,

            "fetch_method":
                "PLAYWRIGHT",

            "fetch_error":
                f"{type(e).__name__}: {e}",
        }

In [45]:
async def collect_article_corpus(
    case_id: str,
    search_hits: pd.DataFrame,
    headless=False,
    checkpoint_every=10,
):
    """
    Search Hit의 original_link를 기준으로
    고유 기사 원문을 수집합니다.

    저장 위치:
        ./stored_data/news/{case_id}/articles.parquet
        ./stored_data/news/{case_id}/raw_html/
    """

    # =====================================================
    # 저장 경로
    # =====================================================

    case_dir = (
        NEWS_STORE_DIR
        / case_id
    )

    raw_html_dir = (
        case_dir
        / "raw_html"
    )


    case_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    raw_html_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    articles_path = (
        case_dir
        / "articles.parquet"
    )


    # =====================================================
    # Unique Article 생성
    # =====================================================

    unique_articles = (
        make_unique_articles(
            search_hits
        )
    )


    print(
        "Search Hit:",
        len(search_hits)
    )

    print(
        "Unique Article:",
        len(unique_articles)
    )


    # =====================================================
    # 기존 Corpus
    # =====================================================

    if articles_path.exists():

        result_df = pd.read_parquet(
            articles_path
        )

    else:

        result_df = pd.DataFrame()


    # SUCCESS인 기사만 skip
    successful_ids = set()


    if (
        not result_df.empty
        and "crawl_status"
        in result_df.columns
    ):

        successful_ids = set(
            result_df.loc[
                result_df[
                    "crawl_status"
                ]
                == "SUCCESS",
                "article_id",
            ]
        )


    # =====================================================
    # HTTP Session
    # =====================================================

    session = requests.Session()


    # =====================================================
    # Playwright
    # =====================================================

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=headless
        )

        context = await browser.new_context(
            locale="ko-KR",
            timezone_id="Asia/Seoul",
        )

        page = await context.new_page()


        processed_since_save = 0


        for idx, row in (
            unique_articles
            .iterrows()
        ):

            article_id = (
                row["article_id"]
            )


            # -----------------------------------------
            # 이미 성공한 기사는 skip
            # -----------------------------------------

            if article_id in successful_ids:

                print(
                    "[SKIP]",
                    article_id
                )

                continue


            url = (
                row["original_link"]
            )


            print()
            print(
                f"[{idx + 1}/"
                f"{len(unique_articles)}]"
            )

            print(
                row["publisher_name"],
                "|",
                row["title"]
            )

            print(
                url
            )


            # =========================================
            # 1차: requests
            # =========================================

            result = (
                download_html_requests(
                    session,
                    url,
                )
            )


            html = result[
                "html"
            ]


            body = (
                extract_article_body(
                    html
                )
            )


            # =========================================
            # 2차: Playwright fallback
            # =========================================

            if not body:

                pw_result = (
                    await download_html_playwright(
                        page,
                        url,
                    )
                )


                if pw_result[
                    "html"
                ]:

                    result = pw_result

                    html = result[
                        "html"
                    ]

                    body = (
                        extract_article_body(
                            html
                        )
                    )


            # =========================================
            # 발행시각
            # =========================================

            (
                published_at,
                published_at_source,
            ) = extract_published_at(
                html
            )


            # =========================================
            # Raw HTML 저장
            # =========================================

            raw_html_path = None


            if html:

                raw_html_path = (
                    raw_html_dir
                    / f"{article_id}.html.gz"
                )


                save_raw_html(
                    html,
                    raw_html_path,
                )


            # =========================================
            # 상태
            # =========================================

            if body:

                crawl_status = (
                    "SUCCESS"
                )

            elif html:

                crawl_status = (
                    "BODY_EXTRACTION_FAILED"
                )

            else:

                crawl_status = (
                    "HTML_DOWNLOAD_FAILED"
                )


            # =========================================
            # 결과 row
            # =========================================

            new_row = {
                "article_id":
                    article_id,

                "case_id":
                    case_id,

                "company":
                    row["company"],

                "publisher_name":
                    row[
                        "publisher_name"
                    ],

                "publisher_domain":
                    row[
                        "publisher_domain"
                    ],

                "title":
                    row["title"],

                "search_description":
                    row[
                        "description"
                    ],

                "original_link":
                    url,

                "final_url":
                    result[
                        "final_url"
                    ],

                "search_hit_count":
                    int(
                        row[
                            "search_hit_count"
                        ]
                    ),

                "published_at":
                    published_at,

                "published_at_source":
                    published_at_source,

                "body":
                    body,

                "body_length":
                    (
                        len(body)
                        if body
                        else 0
                    ),

                "crawl_status":
                    crawl_status,

                "fetch_method":
                    result[
                        "fetch_method"
                    ],

                "fetch_error":
                    result[
                        "fetch_error"
                    ],

                "http_status":
                    result[
                        "http_status"
                    ],

                "raw_html_path":
                    (
                        str(
                            raw_html_path
                        )
                        if raw_html_path
                        else None
                    ),

                "crawled_at":
                    datetime.now(
                        KST
                    ).isoformat(),
            }


            # =========================================
            # 기존 실패 row 교체
            # =========================================

            if not result_df.empty:

                result_df = (
                    result_df[
                        result_df[
                            "article_id"
                        ]
                        != article_id
                    ]
                    .reset_index(
                        drop=True
                    )
                )


            result_df = pd.concat(
                [
                    result_df,
                    pd.DataFrame(
                        [new_row]
                    ),
                ],
                ignore_index=True,
            )


            if crawl_status == "SUCCESS":

                successful_ids.add(
                    article_id
                )


            processed_since_save += 1


            # =========================================
            # Checkpoint
            # =========================================

            if (
                processed_since_save
                >= checkpoint_every
            ):

                result_df.to_parquet(
                    articles_path,
                    index=False,
                )

                processed_since_save = 0


            # 기사 원문 요청 사이 간격
            await asyncio.sleep(
                1.5
            )


        # =================================================
        # 마지막 저장
        # =================================================

        result_df.to_parquet(
            articles_path,
            index=False,
        )


        await context.close()
        await browser.close()


    session.close()


    return result_df

In [188]:
hanmi_articles = (
    await collect_article_corpus(
        case_id="hanmi_20260612",
        search_hits=hanmi_hits,
        headless=False,
    )
)

Search Hit: 5086
Unique Article: 1079

[1/1079]
매일경제 | ‘8천피’ 코앞서 잡주처럼 널뛴 증시…뭐가 끌어내렸나 살펴보니
https://www.mk.co.kr/article/12045327

[2/1079]
녹색경제신문 | 지난해 '네이버' 사던 개인, 올해는 '삼성전자'로…8000피 앞두고 달라진...
https://www.greened.kr/news/articleView.html?idxno=340590

[3/1079]
쿠키뉴스 | “쌓는 기술이 돈 된다”…후공정 반도체 ETF ‘질주’
https://www.kukinews.com/article/view/kuk202605110183

[4/1079]
굿모닝경제 | 코스피, 장중 '7999' 찍고 급락…외국인·기관 매도에 7600선 밀려
https://www.goodkyung.com/news/articleView.html?idxno=286881

[5/1079]
초이스경제 | 外人 폭풍매도, 코스피 '출렁'...LG · 한화는 '껑충'
http://www.choicenews.co.kr/news/articleView.html?idxno=165157

[6/1079]
핀포인트뉴스 | [코스피 지수선물 옵션] 삼성전자·SK하이닉스 급락… 선물시장에 매도...
https://www.pinpointnews.co.kr/news/articleView.html?idxno=452206

[7/1079]
메트로신문 | 한투운용 "RIA 절세 혜택 막차 타세요"…ACE ETF 5종 추천
http://www.metroseoul.co.kr/article/20260512500374

[8/1079]
핀포인트뉴스 | [거래소 기관] 삼성전기 폭풍 매수…반도체·전장주로 다시 몰렸다
https://www.pinpointnews.co.kr/news/articleView.html?idxno=452170

[9/1079]
인사이트코리아 | 한투운용, RIA 절세계좌 투자용 ACE ETF 5종 제시
http

In [189]:
hanmi_articles[
    "crawl_status"
].value_counts(
    dropna=False
)

crawl_status
SUCCESS                   1077
BODY_EXTRACTION_FAILED       2
Name: count, dtype: int64

In [190]:
hanmi_success_rate = (
    hanmi_articles[
        "crawl_status"
    ]
    .eq("SUCCESS")
    .mean()
)

print(
    f"한미반도체 본문 수집 성공률: "
    f"{hanmi_success_rate:.2%}"
)

한미반도체 본문 수집 성공률: 99.81%


In [192]:
print(
    "한미반도체 발행시각 확보율:",
    f"{hanmi_articles['published_at'].notna().mean():.2%}"
)

한미반도체 발행시각 확보율: 95.37%


In [193]:
hanmi_articles[
    hanmi_articles[
        "crawl_status"
    ]
    != "SUCCESS"
][
    [
        "publisher_name",
        "title",
        "original_link",
        "crawl_status",
        "fetch_method",
        "fetch_error",
        "http_status",
    ]
]

,publisher_name,title,original_link,crawl_status,fetch_method,fetch_error,http_status
15,디지털데일리,"日무라타, 반도체 후공정 자동화 투자 확대…국내 관련 기업들도 촉각 ...",https://www.ddaily.co.kr/page/view/2026051209191914125,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,502
650,티브이데일리,"젠슨 황은 왜 '유퀴즈'를 택했나, 접시닦이 소년→시총 1위 CEO 드라마 [...",http://www.tvdaily.co.kr/read.php3?aid=17803706401788642002,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,200


In [47]:
#한미반도체 뉴스 적재 검증
from pathlib import Path
import pandas as pd


def validate_hanmi_news_storage():

    search_hits_path = Path(
        "./news_data/hanmi_20260612/search_hits.parquet"
    )

    articles_path = Path(
        "./stored_data/news/hanmi_20260612/articles.parquet"
    )


    print("=" * 100)
    print("한미반도체 뉴스 적재 검증")
    print("=" * 100)


    # =====================================================
    # 1. 파일 존재 여부
    # =====================================================

    print("\n[1. 파일 존재 여부]")

    print(
        "search_hits.parquet:",
        search_hits_path.exists()
    )

    print(
        "articles.parquet:",
        articles_path.exists()
    )


    if not search_hits_path.exists():
        raise FileNotFoundError(
            search_hits_path
        )

    if not articles_path.exists():
        raise FileNotFoundError(
            articles_path
        )


    # =====================================================
    # 2. 데이터 로드
    # =====================================================

    hits_df = pd.read_parquet(
        search_hits_path
    )

    articles_df = pd.read_parquet(
        articles_path
    )


    # =====================================================
    # 3. 기본 건수
    # =====================================================

    print("\n[2. 기본 건수]")


    unique_hit_urls = set(
        hits_df[
            "original_link"
        ]
        .dropna()
        .unique()
    )


    stored_urls = set(
        articles_df[
            "original_link"
        ]
        .dropna()
        .unique()
    )


    print(
        "Search Hit 수:",
        len(hits_df)
    )

    print(
        "Search Hit의 고유 original_link:",
        len(unique_hit_urls)
    )

    print(
        "articles.parquet 행 수:",
        len(articles_df)
    )

    print(
        "articles의 고유 original_link:",
        len(stored_urls)
    )


    # =====================================================
    # 4. 기사 적재 Coverage
    # =====================================================

    print("\n[3. 기사 적재 Coverage]")


    missing_urls = (
        unique_hit_urls
        - stored_urls
    )


    extra_urls = (
        stored_urls
        - unique_hit_urls
    )


    if unique_hit_urls:

        coverage = (
            len(
                unique_hit_urls
                & stored_urls
            )
            / len(unique_hit_urls)
        )

    else:

        coverage = 0


    print(
        f"Coverage: {coverage:.2%}"
    )

    print(
        "아직 articles에 없는 URL:",
        len(missing_urls)
    )

    print(
        "search_hits에는 없지만 articles에 있는 URL:",
        len(extra_urls)
    )


    # =====================================================
    # 5. 중복 확인
    # =====================================================

    print("\n[4. 중복 확인]")


    duplicate_article_id = (
        articles_df[
            "article_id"
        ]
        .duplicated()
        .sum()
    )


    duplicate_original_link = (
        articles_df[
            "original_link"
        ]
        .duplicated()
        .sum()
    )


    print(
        "중복 article_id:",
        duplicate_article_id
    )

    print(
        "중복 original_link:",
        duplicate_original_link
    )


    # =====================================================
    # 6. Crawl 상태
    # =====================================================

    print("\n[5. Crawl 상태]")


    crawl_status_counts = (
        articles_df[
            "crawl_status"
        ]
        .value_counts(
            dropna=False
        )
    )


    print(
        crawl_status_counts
    )


    success_count = (
        articles_df[
            "crawl_status"
        ]
        .eq("SUCCESS")
        .sum()
    )


    success_rate = (
        success_count
        / len(articles_df)
        if len(articles_df) > 0
        else 0
    )


    print(
        f"\n본문 수집 성공률: "
        f"{success_rate:.2%}"
    )


    # =====================================================
    # 7. SUCCESS인데 body가 비어 있는지
    # =====================================================

    print("\n[6. 본문 무결성]")


    invalid_success = (
        articles_df[
            (
                articles_df[
                    "crawl_status"
                ]
                == "SUCCESS"
            )
            &
            (
                articles_df[
                    "body"
                ]
                .fillna("")
                .str.strip()
                .eq("")
            )
        ]
    )


    print(
        "SUCCESS인데 body가 비어 있는 행:",
        len(invalid_success)
    )


    # =====================================================
    # 8. body_length 검증
    # =====================================================

    body_length_mismatch = (
        articles_df[
            articles_df.apply(
                lambda row:
                    (
                        len(row["body"])
                        if isinstance(
                            row["body"],
                            str
                        )
                        else 0
                    )
                    != row[
                        "body_length"
                    ],
                axis=1,
            )
        ]
    )


    print(
        "body_length 불일치:",
        len(body_length_mismatch)
    )


    # =====================================================
    # 9. 발행시각
    # =====================================================

    print("\n[7. 발행시각]")


    published_count = (
        articles_df[
            "published_at"
        ]
        .notna()
        .sum()
    )


    published_rate = (
        published_count
        / len(articles_df)
        if len(articles_df) > 0
        else 0
    )


    print(
        "published_at 확보 건수:",
        published_count
    )

    print(
        f"published_at 확보율: "
        f"{published_rate:.2%}"
    )


    parsed_dates = pd.to_datetime(
        articles_df[
            "published_at"
        ],
        errors="coerce",
        utc=True,
    )


    invalid_published_at = (
        articles_df[
            articles_df[
                "published_at"
            ].notna()
            &
            parsed_dates.isna()
        ]
    )


    print(
        "파싱 불가능한 published_at:",
        len(invalid_published_at)
    )


    # =====================================================
    # 10. Raw HTML 실제 존재 확인
    # =====================================================

    print("\n[8. Raw HTML]")


    def raw_html_exists(
        path_value
    ):

        if not isinstance(
            path_value,
            str
        ):
            return False

        if not path_value.strip():
            return False

        return Path(
            path_value
        ).exists()


    raw_exists_mask = (
        articles_df[
            "raw_html_path"
        ]
        .apply(
            raw_html_exists
        )
    )


    raw_path_exists_count = (
        articles_df[
            "raw_html_path"
        ]
        .notna()
        .sum()
    )


    print(
        "raw_html_path가 있는 행:",
        raw_path_exists_count
    )

    print(
        "실제 존재하는 raw HTML 파일:",
        raw_exists_mask.sum()
    )

    print(
        "경로는 있지만 파일이 없는 행:",
        (
            articles_df[
                "raw_html_path"
            ]
            .notna()
            &
            ~raw_exists_mask
        ).sum()
    )


    # =====================================================
    # 11. search_hit_count
    # =====================================================

    print("\n[9. Search Hit 연결]")


    invalid_hit_count = (
        articles_df[
            articles_df[
                "search_hit_count"
            ]
            < 1
        ]
    )


    print(
        "search_hit_count < 1:",
        len(invalid_hit_count)
    )


    # =====================================================
    # 12. 최종 구조적 판정
    # =====================================================

    critical_issue_count = (
        len(missing_urls)
        + duplicate_article_id
        + duplicate_original_link
        + len(invalid_success)
        + len(body_length_mismatch)
    )


    print()
    print("=" * 100)


    if critical_issue_count == 0:

        print(
            "[OK] 구조적인 적재 오류는 발견되지 않았습니다."
        )

    else:

        print(
            "[CHECK] 확인이 필요한 구조적 문제가 있습니다."
        )


    print("=" * 100)


    # 상세 검사용 반환
    return {
        "search_hits":
            hits_df,

        "articles":
            articles_df,

        "missing_urls":
            sorted(
                missing_urls
            ),

        "extra_urls":
            sorted(
                extra_urls
            ),

        "invalid_success":
            invalid_success,

        "body_length_mismatch":
            body_length_mismatch,

        "invalid_published_at":
            invalid_published_at,
    }

In [48]:
#한미반도체 뉴스 적재 검증
hanmi_check = (
    validate_hanmi_news_storage()
)

한미반도체 뉴스 적재 검증

[1. 파일 존재 여부]
search_hits.parquet: True
articles.parquet: True

[2. 기본 건수]
Search Hit 수: 5086
Search Hit의 고유 original_link: 1079
articles.parquet 행 수: 1079
articles의 고유 original_link: 1079

[3. 기사 적재 Coverage]
Coverage: 100.00%
아직 articles에 없는 URL: 0
search_hits에는 없지만 articles에 있는 URL: 0

[4. 중복 확인]
중복 article_id: 0
중복 original_link: 0

[5. Crawl 상태]
crawl_status
SUCCESS                   1077
BODY_EXTRACTION_FAILED       2
Name: count, dtype: int64

본문 수집 성공률: 99.81%

[6. 본문 무결성]
SUCCESS인데 body가 비어 있는 행: 0
body_length 불일치: 0

[7. 발행시각]
published_at 확보 건수: 1029
published_at 확보율: 95.37%
파싱 불가능한 published_at: 15

[8. Raw HTML]
raw_html_path가 있는 행: 1079
실제 존재하는 raw HTML 파일: 1079
경로는 있지만 파일이 없는 행: 0

[9. Search Hit 연결]
search_hit_count < 1: 0

[OK] 구조적인 적재 오류는 발견되지 않았습니다.


In [49]:
import pandas as pd


def make_hanmi_validation_sample(
    articles_df: pd.DataFrame,
    random_n=30,
    shortest_n=5,
    longest_n=5,
    top_publishers_n=10,
    random_state=42,
):
    """
    한미반도체 뉴스 본문 품질을 사람이 직접 검증하기 위한 표본 생성.

    구성:
    1. SUCCESS 기사 중 랜덤 30개
    2. 본문 길이가 가장 짧은 5개
    3. 본문 길이가 가장 긴 5개
    4. 기사 수가 많은 상위 10개 언론사에서 1개씩

    중복된 기사는 article_id 기준으로 한 번만 남깁니다.
    """

    # =====================================================
    # SUCCESS 기사만 검증
    # =====================================================

    success_df = (
        articles_df[
            articles_df["crawl_status"] == "SUCCESS"
        ]
        .copy()
        .reset_index(drop=True)
    )

    print("전체 기사 수:", len(articles_df))
    print("SUCCESS 기사 수:", len(success_df))


    if success_df.empty:
        raise ValueError(
            "crawl_status == 'SUCCESS'인 기사가 없습니다."
        )


    samples = []


    # =====================================================
    # 1. 랜덤 표본
    # =====================================================

    random_sample = (
        success_df
        .sample(
            n=min(
                random_n,
                len(success_df)
            ),
            random_state=random_state,
        )
        .copy()
    )

    random_sample["validation_reason"] = "RANDOM"

    samples.append(
        random_sample
    )


    # =====================================================
    # 2. 가장 짧은 본문
    # =====================================================

    shortest_sample = (
        success_df
        .nsmallest(
            min(
                shortest_n,
                len(success_df)
            ),
            "body_length",
        )
        .copy()
    )

    shortest_sample[
        "validation_reason"
    ] = "SHORTEST_BODY"

    samples.append(
        shortest_sample
    )


    # =====================================================
    # 3. 가장 긴 본문
    # =====================================================

    longest_sample = (
        success_df
        .nlargest(
            min(
                longest_n,
                len(success_df)
            ),
            "body_length",
        )
        .copy()
    )

    longest_sample[
        "validation_reason"
    ] = "LONGEST_BODY"

    samples.append(
        longest_sample
    )


    # =====================================================
    # 4. 기사 수가 많은 상위 언론사
    # =====================================================

    top_publishers = (
        success_df[
            "publisher_name"
        ]
        .value_counts()
        .head(
            top_publishers_n
        )
        .index
    )


    publisher_samples = []


    for publisher in top_publishers:

        publisher_df = (
            success_df[
                success_df[
                    "publisher_name"
                ]
                == publisher
            ]
        )


        sample = (
            publisher_df
            .sample(
                n=1,
                random_state=random_state,
            )
            .copy()
        )


        sample[
            "validation_reason"
        ] = "TOP_PUBLISHER"


        publisher_samples.append(
            sample
        )


    if publisher_samples:

        publisher_sample_df = pd.concat(
            publisher_samples,
            ignore_index=True,
        )

        samples.append(
            publisher_sample_df
        )


    # =====================================================
    # 전체 결합
    # =====================================================

    validation_df = pd.concat(
        samples,
        ignore_index=True,
    )


    # 한 기사가 여러 조건에 걸린 경우
    # 어떤 조건으로 선택됐는지 모두 보존
    reason_df = (
        validation_df
        .groupby("article_id")[
            "validation_reason"
        ]
        .agg(
            lambda values:
                ", ".join(
                    sorted(
                        set(values)
                    )
                )
        )
        .rename(
            "validation_reasons"
        )
    )


    validation_df = (
        validation_df
        .drop_duplicates(
            subset=[
                "article_id"
            ],
            keep="first",
        )
        .drop(
            columns=[
                "validation_reason"
            ]
        )
        .merge(
            reason_df,
            on="article_id",
            how="left",
        )
        .reset_index(drop=True)
    )


    # 검토 번호
    validation_df.insert(
        0,
        "validation_no",
        range(
            1,
            len(validation_df) + 1
        ),
    )


    print()
    print(
        "최종 수동 검증 대상:",
        len(validation_df),
        "건"
    )


    return validation_df

In [51]:
articles_df = hanmi_check[
    "articles"
]

hanmi_validation_df = (
    make_hanmi_validation_sample(
        articles_df
    )
)

전체 기사 수: 1079
SUCCESS 기사 수: 1077

최종 수동 검증 대상: 50 건


In [52]:
hanmi_validation_df[
    [
        "validation_no",
        "validation_reasons",
        "publisher_name",
        "title",
        "published_at",
        "body_length",
    ]
]

,validation_no,validation_reasons,publisher_name,title,published_at,body_length
0,1,RANDOM,잡포스트,한미반도체 주가 11% 급등… AI 반도체 수혜 기대에 강세 지속,2026-05-21T14:17:06+09:00,945
1,2,RANDOM,톱스타뉴스,"한미반도체 주가, 5월 27일 장중 322,000원 2.13% 하락",2026-05-27T10:13:55+09:00,644
2,3,RANDOM,조세일보,삼성전자 13% 급등·HPSP 신고가…반도체주 폭풍 랠리 [칩칩폭폭],2026-06-12T10:58:29+09:00,3529
3,4,RANDOM,연합인포맥스,같은 반도체 ETF인데 수익률 '극과 극'…삼전 뛰는데 역주행 레버리지도,2026-05-29T15:12:35+09:00,1914
4,5,RANDOM,연합뉴스,[연합뉴스 이 시각 헤드라인] - 14:30,2026-06-08T14:30:00+09:00,3494
5,6,RANDOM,핀포인트뉴스,반도체 던진 外人 '44조 엑소더스'의 이면...'AI 로봇·전기전자' 판 바...,2026-06-01T17:52:24+09:00,3797
6,7,RANDOM,서울경제,ETF가 주가 움직인다…삼성전기 웃고 한미반도체 울고,2026-06-12T10:15:13+09:00,3303
7,8,RANDOM,PPSS,"한미반도체, 스페이스X에 500억 투자…AI 반도체 수요 '우주로' 확장",2026-06-12T10:23:47+09:00,1347
8,9,RANDOM,톱스타뉴스,"[공매도 브리핑] 5월 29일 거래대금 4조9,242억…종근당·리노공업 상위",2026-05-29T17:38:17+09:00,1627
9,10,RANDOM,한국경제,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다,2026-06-11T16:06:14+09:00,3477


In [53]:
for _, row in hanmi_validation_df.iterrows():

    print("=" * 120)

    print(
        f"[검증 {row['validation_no']}/"
        f"{len(hanmi_validation_df)}]"
    )

    print(
        "선정 이유:",
        row["validation_reasons"]
    )

    print(
        "언론사:",
        row["publisher_name"]
    )

    print(
        "제목:",
        row["title"]
    )

    print(
        "발행시각:",
        row["published_at"]
    )

    print(
        "본문 길이:",
        row["body_length"]
    )

    print(
        "원문 링크:",
        row["original_link"]
    )

    print()
    print("[본문]")
    print(row["body"])
    print()

[검증 1/50]
선정 이유: RANDOM
언론사: 잡포스트
제목: 한미반도체 주가 11% 급등… AI 반도체 수혜 기대에 강세 지속
발행시각: 2026-05-21T14:17:06+09:00
본문 길이: 945
원문 링크: https://www.job-post.co.kr/news/articleView.html?idxno=216541

[본문]
[잡포스트] 전진홍 기자 = 한미반도체 주가가 21일 장중 두 자릿수 상승률을 기록하며 강세 흐름을 이어갔다. 한국거래소(KRX)에 따르면 한미반도체는 이날 오후 2시 16분 기준 전 거래일 대비 3만3천원(11.48%) 오른 32만500원에 거래됐다.
이날 주가는 29만8천원에 출발한 뒤 매수세가 꾸준히 유입되며 상승폭을 확대했다. 장중에는 32만4,500원까지 오르며 강한 탄력을 보였고, 저가는 29만6,500원으로 집계됐다.
거래량은 약 90만주를 기록했으며 거래대금은 2,800억원을 넘어섰다. 시장에서는 반도체 장비 업종 전반에 대한 투자심리 개선과 함께 AI 반도체 시장 확대 기대감이 주가 상승을 이끈 것으로 보고 있다.
특히 고대역폭메모리(HBM) 생산 확대가 핵심 재료로 부각되고 있다. 한미반도체는 반도체 패키징 공정 핵심 장비 분야에서 경쟁력을 보유하고 있어 글로벌 메모리 업체들의 투자 확대 수혜 기대가 커지는 분위기다.
최근 AI 데이터센터와 고성능 컴퓨팅 시장이 빠르게 성장하면서 HBM 관련 장비 수요도 증가하고 있다. 이에 따라 삼성전자와 SK하이닉스 등 주요 고객사의 설비 투자 확대 가능성이 한미반도체 실적 기대감을 높이고 있다는 분석이다.
증권가에서는 반도체 업황 회복과 AI 시장 성장 흐름이 맞물리며 장비업체들의 실적 개선 가능성이 커지고 있다고 평가한다. 특히 첨단 패키징 기술 중요성이 높아지면서 관련 기업들의 기업가치 재평가가 이어질 수 있다는 전망도 나온다.
다만 단기 급등에 따른 변동성 확대 가능성은 경계 요인으로 꼽힌다. 글로벌 반도체 투자 사이클과 미국 통화정책 변화, IT 수요 회복 

In [ ]:
skhynix_articles = (
    await collect_article_corpus(
        case_id="skhynix_20260304",
        search_hits=skhynix_hits,
        headless=False,
    )
)

Search Hit: 64821
Unique Article: 18451

[1/18451]
스카이데일리 | 삼성전자·SK하이닉스 ‘반도체 200억 달러 시대’ 열렸다… 1월 수출 6...
https://m.skyedaily.com/news_view.html?ID=297209

[2/18451]
글로벌이코노믹 | [AI시대 넥스트 리더십]이재용, 직원 다독이고 현장 경영 행보…달라진...
https://www.g-enews.com/view.php?ud=2026020110591684396ed0c62d49_1

[3/18451]
무등일보 | [기고] 반도체 ALD공정 '우뚝'···SK하이닉스 의존 '양날의 검'
http://www.mdilbo.com/detail/AjBTxA/752753

[4/18451]
핀포인트뉴스 | "삼성전자·SK하이닉스 안 부럽다"… 제주반도체, 틈새시장 파고든 '팹리...
https://www.pinpointnews.co.kr/news/articleView.html?idxno=424269

[5/18451]
핀포인트뉴스 | 야간 전기요금 인상, 車 기회지만 반도체·석화·철강은 우려
https://www.pinpointnews.co.kr/news/articleView.html?idxno=424316

[6/18451]
폴리뉴스 | [김능구·황장수의 닥터둠의 전망 1월①] "코스피 5000·코스닥 1000 돌파...
https://www.polinews.co.kr/news/articleView.html?idxno=721478

[7/18451]
경기신문 | 원유철, 경기 남부 ‘K-반도체 벨트’ 특구 지정 제안...“AI 시대 대한...
https://www.kgnews.co.kr/news/article.html?no=883097

[8/18451]
파이낸셜뉴스 | [강남視角] 大주식의 시대
http://www.fnnews.com/news/202602011839356005

[9/18451]
허프포스트코리아 | 1월 무역수지 역대 1월 중 최

In [55]:
skhynix_articles[
    "crawl_status"
].value_counts(
    dropna=False
)

crawl_status
SUCCESS                   18414
BODY_EXTRACTION_FAILED      110
HTML_DOWNLOAD_FAILED          7
Name: count, dtype: int64

In [56]:
skhynix_success_rate = (
    skhynix_articles[
        "crawl_status"
    ]
    .eq("SUCCESS")
    .mean()
)

print(
    f"SK하이닉스 본문 수집 성공률: "
    f"{skhynix_success_rate:.2%}"
)

SK하이닉스 본문 수집 성공률: 99.37%


In [57]:
print(
    "SK하이닉스 발행시각 확보율:",
    f"{skhynix_articles['published_at'].notna().mean():.2%}"
)

SK하이닉스 발행시각 확보율: 93.19%


In [58]:
skhynix_articles[
    skhynix_articles[
        "crawl_status"
    ]
    != "SUCCESS"
][
    [
        "publisher_name",
        "title",
        "original_link",
        "crawl_status",
        "fetch_method",
        "fetch_error",
        "http_status",
    ]
]

,publisher_name,title,original_link,crawl_status,fetch_method,fetch_error,http_status
433,뉴스메이커,1월 다섯째 주 ETF 시장,http://www.newsmaker.or.kr/news/articleView.ht...,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,200
528,CWN,‘워시 쇼크’코스피 5000선 붕괴…올해 첫 매도 사이드카 발동,https://cwn.kr/article/1065598239836141,HTML_DOWNLOAD_FAILED,REQUESTS,"SSLError: HTTPSConnectionPool(host='cwn.kr', p...",None
953,뉴스후플러스,(속보) 삼성전자·SK하이닉스·현대차 모두 '하락',https://www.newswhoplus.com/news/articleView.h...,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,200
1104,뉴데일리,"[핫스탁] SK하이닉스, 프리장서 3% 급락, 90만닉스 붕괴",https://biz.newdaily.co.kr/site/data/html/2026...,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,200
1388,JTBC,"""대감집은 복지도 다르구나""…'억대 성과급'에 '두쫀쿠 팝업'까지",https://news.jtbc.co.kr/article/NB12283232?inf...,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,200
...,...,...,...,...,...,...,...
17973,뉴스1,"""주식 안 할게 한번만 살려 주세요""…12% 폭락한 코스피에 개미들 '패닉...",https://www.news1.kr/finance/general-stock/609...,BODY_EXTRACTION_FAILED,REQUESTS,None,200
18020,뉴데일리,"[핫스탁] 삼전 · SK하이닉스, 에프터서 18만원 · 90만원 회복세",https://biz.newdaily.co.kr/site/data/html/2026...,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,200
18275,JTBC,"""주식 참 쉽죠?"" vs ""내가 사니 나락""…중동 쇼크에 희비 엇갈렸다",https://news.jtbc.co.kr/article/NB12288036?inf...,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,200
18379,JTBC,외국인 투매에 증시 '와르르'…코스피 5800선 붕괴,https://news.jtbc.co.kr/article/NB12287968?inf...,BODY_EXTRACTION_FAILED,PLAYWRIGHT,None,200


In [59]:
#한미반도체 뉴스 적재 검증
from pathlib import Path
import pandas as pd


def validate_skhynix_news_storage():

    search_hits_path = Path(
        "./news_data/skhynix_20260304/search_hits.parquet"
    )

    articles_path = Path(
        "./stored_data/news/skhynix_20260304/articles.parquet"
    )


    print("=" * 100)
    print("SK hynix 뉴스 적재 검증")
    print("=" * 100)


    # =====================================================
    # 1. 파일 존재 여부
    # =====================================================

    print("\n[1. 파일 존재 여부]")

    print(
        "search_hits.parquet:",
        search_hits_path.exists()
    )

    print(
        "articles.parquet:",
        articles_path.exists()
    )


    if not search_hits_path.exists():
        raise FileNotFoundError(
            search_hits_path
        )

    if not articles_path.exists():
        raise FileNotFoundError(
            articles_path
        )


    # =====================================================
    # 2. 데이터 로드
    # =====================================================

    hits_df = pd.read_parquet(
        search_hits_path
    )

    articles_df = pd.read_parquet(
        articles_path
    )


    # =====================================================
    # 3. 기본 건수
    # =====================================================

    print("\n[2. 기본 건수]")


    unique_hit_urls = set(
        hits_df[
            "original_link"
        ]
        .dropna()
        .unique()
    )


    stored_urls = set(
        articles_df[
            "original_link"
        ]
        .dropna()
        .unique()
    )


    print(
        "Search Hit 수:",
        len(hits_df)
    )

    print(
        "Search Hit의 고유 original_link:",
        len(unique_hit_urls)
    )

    print(
        "articles.parquet 행 수:",
        len(articles_df)
    )

    print(
        "articles의 고유 original_link:",
        len(stored_urls)
    )


    # =====================================================
    # 4. 기사 적재 Coverage
    # =====================================================

    print("\n[3. 기사 적재 Coverage]")


    missing_urls = (
        unique_hit_urls
        - stored_urls
    )


    extra_urls = (
        stored_urls
        - unique_hit_urls
    )


    if unique_hit_urls:

        coverage = (
            len(
                unique_hit_urls
                & stored_urls
            )
            / len(unique_hit_urls)
        )

    else:

        coverage = 0


    print(
        f"Coverage: {coverage:.2%}"
    )

    print(
        "아직 articles에 없는 URL:",
        len(missing_urls)
    )

    print(
        "search_hits에는 없지만 articles에 있는 URL:",
        len(extra_urls)
    )


    # =====================================================
    # 5. 중복 확인
    # =====================================================

    print("\n[4. 중복 확인]")


    duplicate_article_id = (
        articles_df[
            "article_id"
        ]
        .duplicated()
        .sum()
    )


    duplicate_original_link = (
        articles_df[
            "original_link"
        ]
        .duplicated()
        .sum()
    )


    print(
        "중복 article_id:",
        duplicate_article_id
    )

    print(
        "중복 original_link:",
        duplicate_original_link
    )


    # =====================================================
    # 6. Crawl 상태
    # =====================================================

    print("\n[5. Crawl 상태]")


    crawl_status_counts = (
        articles_df[
            "crawl_status"
        ]
        .value_counts(
            dropna=False
        )
    )


    print(
        crawl_status_counts
    )


    success_count = (
        articles_df[
            "crawl_status"
        ]
        .eq("SUCCESS")
        .sum()
    )


    success_rate = (
        success_count
        / len(articles_df)
        if len(articles_df) > 0
        else 0
    )


    print(
        f"\n본문 수집 성공률: "
        f"{success_rate:.2%}"
    )


    # =====================================================
    # 7. SUCCESS인데 body가 비어 있는지
    # =====================================================

    print("\n[6. 본문 무결성]")


    invalid_success = (
        articles_df[
            (
                articles_df[
                    "crawl_status"
                ]
                == "SUCCESS"
            )
            &
            (
                articles_df[
                    "body"
                ]
                .fillna("")
                .str.strip()
                .eq("")
            )
        ]
    )


    print(
        "SUCCESS인데 body가 비어 있는 행:",
        len(invalid_success)
    )


    # =====================================================
    # 8. body_length 검증
    # =====================================================

    body_length_mismatch = (
        articles_df[
            articles_df.apply(
                lambda row:
                    (
                        len(row["body"])
                        if isinstance(
                            row["body"],
                            str
                        )
                        else 0
                    )
                    != row[
                        "body_length"
                    ],
                axis=1,
            )
        ]
    )


    print(
        "body_length 불일치:",
        len(body_length_mismatch)
    )


    # =====================================================
    # 9. 발행시각
    # =====================================================

    print("\n[7. 발행시각]")


    published_count = (
        articles_df[
            "published_at"
        ]
        .notna()
        .sum()
    )


    published_rate = (
        published_count
        / len(articles_df)
        if len(articles_df) > 0
        else 0
    )


    print(
        "published_at 확보 건수:",
        published_count
    )

    print(
        f"published_at 확보율: "
        f"{published_rate:.2%}"
    )


    parsed_dates = pd.to_datetime(
        articles_df[
            "published_at"
        ],
        errors="coerce",
        utc=True,
    )


    invalid_published_at = (
        articles_df[
            articles_df[
                "published_at"
            ].notna()
            &
            parsed_dates.isna()
        ]
    )


    print(
        "파싱 불가능한 published_at:",
        len(invalid_published_at)
    )


    # =====================================================
    # 10. Raw HTML 실제 존재 확인
    # =====================================================

    print("\n[8. Raw HTML]")


    def raw_html_exists(
        path_value
    ):

        if not isinstance(
            path_value,
            str
        ):
            return False

        if not path_value.strip():
            return False

        return Path(
            path_value
        ).exists()


    raw_exists_mask = (
        articles_df[
            "raw_html_path"
        ]
        .apply(
            raw_html_exists
        )
    )


    raw_path_exists_count = (
        articles_df[
            "raw_html_path"
        ]
        .notna()
        .sum()
    )


    print(
        "raw_html_path가 있는 행:",
        raw_path_exists_count
    )

    print(
        "실제 존재하는 raw HTML 파일:",
        raw_exists_mask.sum()
    )

    print(
        "경로는 있지만 파일이 없는 행:",
        (
            articles_df[
                "raw_html_path"
            ]
            .notna()
            &
            ~raw_exists_mask
        ).sum()
    )


    # =====================================================
    # 11. search_hit_count
    # =====================================================

    print("\n[9. Search Hit 연결]")


    invalid_hit_count = (
        articles_df[
            articles_df[
                "search_hit_count"
            ]
            < 1
        ]
    )


    print(
        "search_hit_count < 1:",
        len(invalid_hit_count)
    )


    # =====================================================
    # 12. 최종 구조적 판정
    # =====================================================

    critical_issue_count = (
        len(missing_urls)
        + duplicate_article_id
        + duplicate_original_link
        + len(invalid_success)
        + len(body_length_mismatch)
    )


    print()
    print("=" * 100)


    if critical_issue_count == 0:

        print(
            "[OK] 구조적인 적재 오류는 발견되지 않았습니다."
        )

    else:

        print(
            "[CHECK] 확인이 필요한 구조적 문제가 있습니다."
        )


    print("=" * 100)


    # 상세 검사용 반환
    return {
        "search_hits":
            hits_df,

        "articles":
            articles_df,

        "missing_urls":
            sorted(
                missing_urls
            ),

        "extra_urls":
            sorted(
                extra_urls
            ),

        "invalid_success":
            invalid_success,

        "body_length_mismatch":
            body_length_mismatch,

        "invalid_published_at":
            invalid_published_at,
    }

In [64]:
#한미반도체 뉴스 적재 검증
skhynix_check = (
    validate_skhynix_news_storage()
)

SK hynix 뉴스 적재 검증

[1. 파일 존재 여부]
search_hits.parquet: True
articles.parquet: True

[2. 기본 건수]
Search Hit 수: 64821
Search Hit의 고유 original_link: 18451
articles.parquet 행 수: 18451
articles의 고유 original_link: 18451

[3. 기사 적재 Coverage]
Coverage: 100.00%
아직 articles에 없는 URL: 0
search_hits에는 없지만 articles에 있는 URL: 0

[4. 중복 확인]
중복 article_id: 0
중복 original_link: 0

[5. Crawl 상태]
crawl_status
SUCCESS                   18334
BODY_EXTRACTION_FAILED      110
HTML_DOWNLOAD_FAILED          7
Name: count, dtype: int64

본문 수집 성공률: 99.37%

[6. 본문 무결성]
SUCCESS인데 body가 비어 있는 행: 0
body_length 불일치: 0

[7. 발행시각]
published_at 확보 건수: 17193
published_at 확보율: 93.18%
파싱 불가능한 published_at: 406

[8. Raw HTML]
raw_html_path가 있는 행: 18444
실제 존재하는 raw HTML 파일: 18444
경로는 있지만 파일이 없는 행: 0

[9. Search Hit 연결]
search_hit_count < 1: 0

[OK] 구조적인 적재 오류는 발견되지 않았습니다.
